# Flow modality

`HyperNetSpartan` with a `FlowVAE` weight generator, trained on the shapes task. Instead of two
explicit conditions (as in `basic_multimodality`), the attention weights are sampled from a
normalizing flow, and each sample is a candidate solution.

The model comes from `src/`, with autoreload on, so edits there take effect when the model is rebuilt.
The training loop below is a stripped-down copy of `HyperNetSpartan.fit`.

In [ ]:
%load_ext autoreload
%autoreload 2

import random
from functools import partial

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import display
from tqdm.auto import tqdm

from sparse_generalization.utils.dataloading import get_shapes_datasets
from sparse_generalization.utils.util_funcs import build_lr_scheduler
from sparse_generalization.models.hypernet import HyperNetSpartan
from sparse_generalization.layers.vae import FlowVAE
from sparse_generalization.layers.priors import make_unit_gaussian, make_disconnected_prior
from sparse_generalization.layers.diversity_losses import CosineRepDiv

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

SET_NAMES = ["id", "a", "b"]
BASE_PRIORS = {"gaussian": make_unit_gaussian, "disconnected": make_disconnected_prior}


class Args:
    # data
    width: int = 3
    num_train: int = 8000
    batch_size: int = 256

    # training (mirrors HyperNetSpartan.fit)
    epochs: int = 200
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.999
    lr_decay: str = "linear"      # 'none' | 'linear'
    lr_warmup: bool = False
    beta: float = 1e-2            # weight on the gen loss, log q(w) - log p(w)
    div_coeff: float = 0.0        # CosineRepDiv over the sampled weight sets; needs num_modes > 1
    val_every: int = 10

    # model (defaults from config/model/hypernet.yaml)
    loss: str = "ce"              # 'ce' | 'bce'
    dim: int = 16
    num_layers: int = 1
    hyper_type: str = "qk"        # 'qk' | 'mha' | 'mask'
    residual: bool = True
    layernorm: bool = False
    pe_type: str = "sin"

    # flow weight generator
    base_prior: str = "gaussian"  # 'gaussian' | 'disconnected' (one mode per sample, pins samples to num_modes)
    n_flows: int = 3
    flow_hidden: tuple = (32, 32)
    prior_type: str = "nf"        # 'uniform' | 'normal' | 'laplace' | 'nf'
    prior_n_flows: int = 3
    prior_hidden: tuple = (32, 32)
    num_modes: int = 1            # weight sets sampled per training step
    num_eval_samples: int = 5     # weight sets sampled at eval

    seeds: tuple = (0,)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

### Data

In [ ]:
def load_data(args, seed=0):
    # The loader splits num_train evenly across the two classes, so it has to be even.
    n = args.num_train + args.num_train % 2
    train_set, val_sets, test_sets, _ = get_shapes_datasets(n, "../data/shapes", args.width, False)
    train_loader = DataLoader(train_set, args.batch_size, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    return train_loader, dict(zip(SET_NAMES, val_sets)), dict(zip(SET_NAMES, test_sets))

### Model

In [ ]:
def build_model(args):
    weight_gen = partial(
        FlowVAE,
        prior_func=BASE_PRIORS[args.base_prior],
        flow_params={"n_flows": args.n_flows, "hidden_features": list(args.flow_hidden)},
    )
    return HyperNetSpartan(
        out_dim=1 if args.loss == "bce" else 2,
        loss_type=args.loss,
        weight_gen=weight_gen,
        prior_type=args.prior_type,
        prior_params={"n_flows": args.prior_n_flows, "hidden_features": list(args.prior_hidden)},
        num_mha_layers=args.num_layers,
        include_agg_layer=True,
        seq_len=args.width ** 2,
        num_embeddings=args.width ** 2,
        embedding_inp=True,
        model_dim=args.dim,
        hyper_type=args.hyper_type,
        residual=args.residual,
        layernorm=args.layernorm,
        pe_type=args.pe_type,
        num_modes=args.num_modes,
        num_eval_samples=args.num_eval_samples,
        div_coeff=args.div_coeff,
        div_loss=CosineRepDiv,
        beta=args.beta,
        lr=args.lr,
        beta1=args.beta1,
        beta2=args.beta2,
        device=DEVICE,
    ).to(DEVICE)


@torch.no_grad()
def evaluate(model, dataset, seed=0):
    was_training = model.training
    model.eval()
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        x, y = dataset.x.to(DEVICE), dataset.y.to(DEVICE)
        probs, _, _ = model(x, evaluate=True, ret_mean=False)  # (e, b, c)
    crit = model.criterion
    per_sample = np.array([crit.accuracy_from_probs(p, y).item() for p in probs])
    ensemble = crit.accuracy_from_probs(probs.mean(0), y).item()
    model.train(was_training)
    return per_sample, ensemble

### Train

In [ ]:
def train(args, seed=0):
    seed_everything(seed)
    train_loader, val_sets, test_sets = load_data(args, seed)
    model = build_model(args)
    optimizer = model.optimizer
    scheduler = build_lr_scheduler(optimizer, args.epochs * len(train_loader), args.lr_decay, args.lr_warmup)
    crit = model.criterion
    e = model.hyper_net.effective_evals(model.num_modes)

    hist = {k: [] for k in ("loss", "acc", "gen", "div")}
    val_hist = {k: [] for k in SET_NAMES}   # (ensemble, best sample) per val step
    val_epochs = []
    fig, (ax_acc, ax_loss, ax_gen) = plt.subplots(1, 3, figsize=(15, 4))
    handle = None

    pbar = tqdm(range(args.epochs), desc=f"seed {seed}")
    for epoch in pbar:
        model.train()
        sums = dict.fromkeys(hist, 0.0)
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            # Same objective as HyperNetSpartan.fit: task loss over the e sampled weight sets,
            # plus beta * (log q(w) - log p(w)) and the diversity term.
            out, masks, log_q, log_p, attns, div = model(x)
            out = out.view(e, -1, model.out_dim)            # (e, b, c)
            y_e = y.unsqueeze(0).expand(e, -1, -1)          # (e, b, 1)
            rec_loss = crit.loss(out, y_e)
            gen_loss = (log_q - log_p).mean()
            loss = rec_loss + args.beta * gen_loss + args.div_coeff * div.squeeze()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            sums["loss"] += rec_loss.item()
            sums["gen"] += gen_loss.item()
            sums["div"] += div.item()
            sums["acc"] += crit.accuracy(out.detach(), y_e).item()

        for k in hist:
            hist[k].append(sums[k] / len(train_loader))

        status = f"L {hist['loss'][-1]:.3f} | acc {hist['acc'][-1]:.3f} | gen {hist['gen'][-1]:.2f}"
        if args.div_coeff:
            status += f" | div {hist['div'][-1]:.3f}"

        if epoch % args.val_every == 0 or epoch == args.epochs - 1:
            val_epochs.append(epoch)
            for name, ds in val_sets.items():
                per_sample, ens = evaluate(model, ds)
                val_hist[name].append((ens, per_sample.max()))
            status += " | val " + " ".join(f"{k} {v[-1][1]:.2f}" for k, v in val_hist.items())

            ax_acc.cla()
            ax_acc.plot(hist["acc"], lw=1.5, color="k", label="train")
            for name, v in val_hist.items():
                line, = ax_acc.plot(val_epochs, [b for _, b in v], marker="o", ms=3, label=f"val {name} (best sample)")
                ax_acc.plot(val_epochs, [m for m, _ in v], ls="--", color=line.get_color(), alpha=0.6)
            ax_acc.set_title("Accuracy (dashed: ensemble)")
            ax_acc.set_ylim(0, 1.02)
            ax_acc.legend(loc="lower right", frameon=False, fontsize=8)
            ax_loss.cla(); ax_loss.plot(hist["loss"]); ax_loss.set_title(f"Train {args.loss.upper()} loss")
            ax_gen.cla(); ax_gen.plot(hist["gen"]); ax_gen.set_title("Gen loss  log q - log p")
            for a in (ax_acc, ax_loss, ax_gen):
                a.set_xlabel("epoch"); a.grid(alpha=0.3); a.spines[["top", "right"]].set_visible(False)
            fig.suptitle(f"seed {seed}  ·  epoch {epoch + 1}/{args.epochs}")
            fig.tight_layout()
            if handle is None:
                handle = display(fig, display_id=True)
            else:
                handle.update(fig)

        pbar.set_postfix_str(status, refresh=False)

    plt.close(fig)
    model.eval()
    results = {name: evaluate(model, ds) for name, ds in test_sets.items()}
    return model, results, hist

### Run

In [ ]:
args = Args()
args.epochs = 200
args.num_modes = 1
args.num_eval_samples = 5
args.beta = 1e-2
args.div_coeff = 0.0

runs = {}
for seed in args.seeds:
    model, results, hist = train(args, seed=seed)
    runs[seed] = results
    print(f"\nseed {seed}")
    print(f"{'set':>4} | {'ensemble':>8} | {'best':>5} | per-sample")
    for name, (per_sample, ens) in results.items():
        print(f"{name:>4} | {ens:>8.3f} | {per_sample.max():>5.3f} | {np.round(per_sample, 3)}")